In [1]:
%load_ext autoreload
%autoreload 2

from kontext.visualization import get_figs,space_plot,space_plot_mask,get_proportions,plot_proportions,heatmap_proportions,plot_ccc_score_per_group,umap_plot,plot_ct_pair_across_samples,plot_region_distribution,pie_region
from utils.load_data import load_adata,load_and_preprocess_adata
from kontext.data import merge_adatas,preprocess_adata
from kontext.kontext_class import Kontext
from kontext.utils import local_alignment, get_LR_pairs, pca_of_mat,get_purity_vector_per_domain


from umap.umap_ import UMAP
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# plt.style.use('dark_background')
plt.rcParams['figure.facecolor'] = '#fcf8f500'
plt.rcParams['axes.facecolor'] = '#fcf8f500'

from itables import init_notebook_mode
init_notebook_mode(all_interactive=True)



In [2]:
dataset = 'CARDIO'
samples = ['sham_S1','sham_S2',
            'cryo7D_S1','cryo7D_S2','cryo7D_S3',        
            'cryo28D_S1','cryo28D_S2','cryo28D_S3',
            ]


## Supplementary table 3 - Refined cell types

In [ ]:
ctcryo ={
        'unknown':'other',
        'Tumor':'Tumor cell',
        'Plasma_IgG':'Plasma IgG',
        'CAF':'CAF',
        'Fibroblasts':'Fibroblast',
        'T_cell':'T cells',
        'CAM':'CAM',
        'Macrophages':'Macrophage',
        'High_Endothelial_Venules':'other',
        'M1_macrophages':'Macrophage',
        'Plasma_IgA':'Plasma IgA',
        'Germinal_Center_Plasma_IgM_B_cell':'Plasma IgM',
        'Tumor_Keratin_Pearl':'Keratin Pearl',
        'Dendritic_cells':'other',
        'Cytotoxic_IFN_signaling':'other',
        'Mast_cells':'other',
        'Cortex_CCL21':'other'}

pd.DataFrame(ctcryo.items(),columns = ['original','refined']).to_csv('./kontext_reproducibility/figures/cryo/table_cell_types.csv',index=True)

## Spatial domains computation

In [3]:
no_preprocessing = {'preprocessing':None}
preprocessing = {'preprocessing':'image'}
embedding = {"sigma_e":1, "snn":20, 'alpha':0.5}
spatial_domains = {'clustering':'gmm','seed':0,'n_domains':8}
niche,me,communication=True,True,True
# niche,me,communication=False,False,False



adatas = {}
for k in samples:
    print(k)
    adata = load_and_preprocess_adata({'dataset':'CARDIO','sample':k},preprocessing_params=no_preprocessing)
    adata.layers['counts'] = adata.X
    adata = preprocess_adata(adata,**preprocessing)

    adata.uns['data']['sample']=k
    ko = Kontext(**embedding,verbose=False)
    ko.weights(adata)
    ko.fit(adata,add_layer=True)
    ko.smooth_and_pca(adata, "wc", "C",add_layer=True)


    if niche:
        ko.niche(adata, cell_type_key="global_cell_type",verbose=False)
        ko.niche(adata, cell_type_key="cell_type",verbose=False)
    if me:
        ko.local_alignment(adata, layer_C="C")
    adatas[k] = adata 
adata_ = merge_adatas(adatas,block_diag_keys=['w','ws','wc','we'])

adata_.uns['data'] = {'dataset':'CARDIO','sample':'all','pp_str':'sparse_hvg_joint'}
ko = Kontext(**embedding,verbose=True)
print('spatial domains')
ko.spatial_domains_of_merged_adata(adata_,**spatial_domains)




## Labels and colors 

In [5]:

spatial_domains_mapping = {
    3:0,
    1:1,
    2:2,
    4:6,
    0:7,
    7:4,
    5:3,
    6:5,}


adata_.obs['spatial_domains_sorted'] = adata_.obs['spatial_domains'].map(spatial_domains_mapping)
adata_.obs['spatial_domains_sorted'] = adata_.obs['spatial_domains_sorted'].astype('category').cat.set_categories(sorted(spatial_domains_mapping.keys()))


print(adata_.obs['main_cell_type'].nunique())
spatial_domains_colors = [
    "#4ad66d","#2A835F","#054a29","#56cfe1",
    "#7692FF","#6930c3","#ffcad4","#853953",]
# spatial_domains_colors = [
#     "#0072B2","#D55E00","#009E73","#CC79A7",
#     "#E69F00","#56B4E9","#F0E442","#4E1A1A",]
cell_type_colors = [
    "#8DD3C7","#FFFFB3","#BEBADA","#FB8072",
    "#80B1D3","#FDB462","#B3DE69","#FCCDE5",
    "#D9D9D9","#BC80BD","#CCEBC5", "#FFED6F",
    "#DCD685","#D7AFAF",

    ]
time_colors = ["#d7b043","#BE4A4A","#006267",]

adata_.uns['spatial_domains_colors'] = spatial_domains_colors
adata_.uns['spatial_domains_sorted_colors'] = spatial_domains_colors
adata_.uns['main_cell_type_colors'] = cell_type_colors
adata_.uns['time_colors'] = time_colors


samples_map = {
    'sham_S1':'sham1',
    'sham_S2':'sham2',
    'cryo7D_S1':'7D1',
    'cryo7D_S2':'7D2',
    'cryo7D_S3':'7D3',
    'cryo28D_S1':'28D1',
    'cryo28D_S2':'28D2',
    'cryo28D_S3':'28D3',
    }

adata_.obs['sample'] = adata_.obs['sample'].map(samples_map)
space_plot(adata_,'sample')

## Figure 3B

In [6]:
color = "spatial_domains_sorted"
domains = adata_.obs[color].sort_values().unique()
sample='7D1'

ncols=len(domains)


adata = adata_[adata_.obs['sample']==sample]

group_names = ['Fibroblasts and Myeloids','Vascular','Cardiomyocytes']
groups = [0,3,6],[1,4,7],[2,5,8]
groups = [0,1,2,6],[3,4,5,7]


for i,group in enumerate(groups):
    fig,axes = get_figs(4,4,5)
    for ax,domain in zip(axes,group):
        if domain == 8:
            ax.set_axis_off()
            continue
        for spine in ax.spines.values():
            spine.set_visible(False)
        mask = space_plot_mask(adata,color,domain,ax=ax,snm=5,sm=50)
        ax.set_title('',fontsize=20)        
        ax.set_xlabel(f"SD{domain}",fontsize=40,color=adata.uns[f'{color}_colors'][domain])
        ax.set_ylabel("")
        ax.get_legend().remove()

## Figures 3C and 3D 

In [7]:
col1,col2 = ['spatial_domains_sorted','main_cell_type']
fig,axes = get_figs(2,7,5)
fs = 15

ax = axes[0]
proportions1 = get_proportions(adata_,col1,col2,normalize=False)
print(proportions1.sum(1))
print(proportions1.shape)
display(proportions1)
proportions1 = get_proportions(adata_,col1,col2,normalize=False)
plot_region_distribution(proportions1,colors = adata_.uns[f'{col2}_colors'] ,ax=ax,fontsize=12,format_value='.2f',threshold=.1,percentage=True)
ax.get_legend().remove()
for tl in ax.get_yticklabels():
    c = int(tl._text)
    tl.set_color(adata_.uns[f'{col1}_colors'][c])
ax.set_yticklabels([f'SD{tick._text}' for tick in ax.get_yticklabels()], rotation='horizontal', fontsize=fs) 
ax.set_xticklabels([f'{xtick._text}' for xtick in ax.get_xticklabels()], rotation='horizontal', fontsize=15) 

# ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=fs) 
ax.set_title('Domain compositions',fontsize=fs)
# ax.set_ylabel('Proportions',fontsize=fs)
ax.set_xlabel('',fontsize=fs)






ax = axes[1]
cols = (['sample','main_cell_type'],)#,['time','main_cell_type'])#['sample','global_cell_type'])
for domain in range(8):
    if domain in [0]:
        adata = adata_[adata_.obs['spatial_domains_sorted']==domain]
        for col1,col2 in cols:

            # fig,axes = get_figs(1,6,5)
            proportions2 = get_proportions(adata,col1,col2,normalize='index')
            proportions2 = get_proportions(adata,col1,col2,normalize=False)
            ax.set_title(f'domain {domain}')
            plot_proportions(proportions2,ax=ax,colors =adata_.uns[f'{col2}_colors'])
            ax.set_xticklabels([xtick._text.replace('_S','').replace('cryo','')#.replace('sham','0D') 
                                for xtick in ax.get_xticklabels()], rotation='vertical', fontsize=fs) 
            ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=fs) 
            ax.set_title('SD0 over time',fontsize=fs)
            ax.set_ylabel('$n_{obs}$',fontsize=fs)
            ax.set_xlabel('',fontsize=fs)
            legend = ax.legend(fontsize=15,bbox_to_anchor=(1.05, 1.05),loc='upper left',frameon=False)
fig.tight_layout()

## Figure 3E 

### Computing purity 

In [8]:
adata1s = {}
purity = {}
for cluster in adata_.obs['spatial_domains_sorted'].unique():
    # adata1 = adata_[adata_.obs['clustering'].isin(cluster)]

    adata1 = adata_[adata_.obs['spatial_domains_sorted']==cluster]
    for layer in 'XCE':
        print(layer, cluster,end=' ')
        mat = adata1.X if layer == 'X' else adata1.layers[layer]  if layer in 'CE' else adata1.obsm[f'niche_global_cell_type'] if layer == 'N' else adata1.obsm[f'niche_cell_type']
        print(mat.shape)
        n_comps = 10 if layer in 'NM' else 50
        pca_of_mat(adata1, mat, n_comps=n_comps, key=f"{layer}{cluster}")
        pca = adata1.obsm[f'{layer}{cluster}_pca']
        adata1.obsm[f'{layer}{cluster}_umap'] = UMAP(n_components=2).fit_transform(pca)


    # for layer in 'XCE':#EN':
        print('purity')
        ks, cp = get_purity_vector_per_domain(
            adata1.obsm[f'{layer}{cluster}_pca'],
            adata1.obs['time'].values,
            adata1.obs['spatial_domains_sorted'].values,
        )
        purity[f'{layer}{cluster}'] = {'ks': ks, 'domain_purities': cp}
    adata1s[f'{cluster}'] = adata1

### Plot Figure 3E

In [9]:

from matplotlib.patches import Patch
layers = 'E'

# for i, layer in enumerate('XCEN'):
for i, layer in enumerate(layers):
    fig, ax = get_figs(1,8,5)
    # if isinstance(axes,np.ndarray):
    #     ax = axes[i]
    # else:
    #     ax = axes
    handles = []
    for domain in range(8):
        key = f'{layer}{domain}'
        if key in purity:
            ks = purity[key]['ks']
            # umap_plot(adata,color='time',umap_key=f'{layer}{name}_umap')
            arr = np.array(list(purity[key]['domain_purities'].values()))  # (n_domains, n_ks)
            mean = arr.mean(axis=0)
            # lo, hi = arr.min(axis=0), arr.max(axis=0)
            color = adata_.uns['spatial_domains_sorted_colors'][domain]
            ax.plot(ks, mean, color=color)
            handles.append(Patch(facecolor=color, label=f'SD{domain}'))
            # ax.fill_between(ks, lo, hi, alpha=0.2, color=color)
    # legend = ax.legend(fontsize=15,bbox_to_anchor=(1.05, 1),loc='upper left')
    legend = ax.legend(handles=handles, fontsize=20, bbox_to_anchor=(1.05, 1), loc='upper left',frameon=False)
    # For text in legend 
    # for text in legend: 
    #     change text color. 
    # ax.get_legend().remove()
    ax.set_ylim(0, 1)
    ax.set_title(f'Temporal concentration in {layer}',fontsize=25)
    ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
    ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=15) 
    ax.set_xlabel('$n$ neighbors',fontsize=15)
    ax.set_ylabel('Purity score',fontsize=15)
    fig.tight_layout()
    # fig.savefig(f'kontext_project/figures/cryo/cryo_2f_{layer}_purity.pdf',bbox_inches='tight')

## Figures 3F and 3G

In [10]:
fig, axes = get_figs(2, 6, 5)

from kontext.utils import lr_normalize,lr_moments,lr_availability,lr_moments,lr_score,lr_aggregate,lr_cell_score,lr_log

mode = 'mean'
pairs = get_LR_pairs(adata_, organism='mouse', verbose=False)


lr_normalize(adata_, pairs)
lr_moments(adata_, 'X_lr')
lr_availability(adata_, None)
lr_moments(adata_, 'A_all')
lr_cell_score(adata_, 'all', mode=mode, col='LR_A_all',src='A')  


ax = plot_ccc_score_per_group(adata_, 'spatial_domains_sorted', 'LR_A_all','sample','7D1', ax=axes[0], insersion=(.10, .12))
ax.set_title('Communication scores', fontsize=25)
ax.set_ylabel('Mean per domain', fontsize=15)
ax.set_xlabel(''); ax.set_yticks([])
ax.set_xticklabels([f'SD{t._text}' for t in ax.get_xticklabels()], rotation='horizontal', fontsize=15)
ax = plot_ccc_score_per_group(adata_, 'spatial_domains_sorted', 'local_alignment', 'sample','7D1', ax=axes[1], insersion=(.10, .12))
ax.set_title('Local alignment scores', fontsize=25)
ax.set_ylabel('')
ax.set_xlabel(''); ax.set_yticks([])
ax.set_xticklabels([f'SD{t._text}' for t in ax.get_xticklabels()], rotation='horizontal', fontsize=15)
fig.tight_layout()

gap = 0.06  
for ax in axes[:1]:  
    p = ax.get_position()
    ax.set_position([p.x0 - gap/2, p.y0, p.width, p.height])
for ax in axes[1:]:   
    p = ax.get_position()
    ax.set_position([p.x0 + gap/2, p.y0, p.width, p.height])
plt.show()

## Figure 3H

### Compute communication scores 

In [11]:
import seaborn as sns
groups, ts, tr = 'main_cell_type', 'Fibroblast', 'Macrophage'
mode, samples_ = 'mean', ['sham_S1', 'cryo7D_S1', 'cryo28D_S1']
mode, times_ = 'mean', ['sham', 'cryo7D', 'cryo28D']  




mode = 'mean'
pairs = get_LR_pairs(adata_, organism='mouse', verbose=False)



lr_normalize(adata_, pairs)
lr_availability(adata_, None)
lr_moments(adata_, 'X_lr')
lr_moments(adata_, 'A_all')
lr_cell_score(adata_, 'all', mode=mode, col='LR_all')  
lr_moments(adata_, 'P_all')


ts, tr = 'Fibroblast', 'Macrophage'
for tx,tc in (ts, tr),(tr,ts),(ts, ts), (tr, tr): 
    # lr_moments(adata_, f'A_{tc}', om=((adata_.obs[f'm_{tc}'] > 0) & (adata_.obs['main_cell_type']==tx) ).values,r=r)
    lr_availability(adata_, tx, groups)
    lr_availability(adata_, tc, groups)
    r = {'col':groups,'value':tx}
    lr_moments(adata_, 'X_lr',r=r)
    lr_moments(adata_, f'A_{tc}', om=(adata_.obs[f'm_{tc}'] > 0).values,r=r)
    lr_moments(adata_, f'P_{tc}', om=(adata_.obs[f'm_{tc}'] > 0).values,r=r)


### Plot figure 3H 

In [14]:
genes = [
    'Cfh_Itgam','C3_Itgam','C3_C3ar1',
    'Ccl12_Ccr2','Spp1_Cd44','Mmp9_Cd44','Ptn_Sdc1',
    'Thbs4_Cd36','Thbs4_Sdc1','Comp_Sdc1',
    'Bmp7_Eng',
    'Timp3_Cd44','S100a4_Erbb2',
]

send_groups = [
    [(ts, tr), (tr, tr)],   
    [(tr, ts), (ts, ts)],  
]

top=15
fs = 12
fig, axes = get_figs(len(send_groups), 12, 6)  

for i, (ax, pair_list) in enumerate(zip(axes, send_groups)):
    parts, senders = [], []
    for (s_, r_) in pair_list:
        r = {'col': groups, 'value': r_}
        S = lr_score(adata_, s_, r_, groups, mode, r=r,src='P')

        gi = pd.Index(adata_.uns['lr']['genes'])
        pos = adata_.obs_names.get_indexer(S.index)
        det = pd.Series(np.asarray((adata_.obsm['X_lr'][pos][:, gi.get_indexer(pairs.receptor)] > 0).mean(0)).ravel(), index=pairs.index)
        keep = det[det > .1].index
        T = lr_aggregate(adata_, S, s_, by='time')['all'].reindex(times_)[[g for g in genes if g in keep and g in S.columns]]
        parts.append((lr_log(T) if mode == 'mean' else T).T)
        senders.append(s_)

    merged = pd.concat(parts, axis=1)   

    label = 'Sender-receiver score' if i == 1 else ''
    cbar = (i == 1)
    print(merged)
    hm = sns.heatmap(merged, ax=ax, cmap='RdBu_r', center=0, vmin=-6, vmax=4,
                annot=True, fmt='.2f', annot_kws={'fontsize':15},
                linewidths=.5, cbar=cbar, cbar_kws={'label': label})
    if cbar:
        cbar_ax = hm.collections[0].colorbar
        cbar_ax.set_label(cbar_ax.ax.get_ylabel(), fontsize=fs)

    ax.axvline(len(parts[0].columns), color='white', lw=12)   # separator between the 2 senders

    sec = ax.secondary_xaxis('top')
    sec.set_xticks([1.5, len(parts[0].columns) + 1.5])
    sec.set_xticklabels([f'{s} to {r}' for s,r in pair_list], fontsize=20)
    sec.tick_params(length=0)
    ax.set_xlabel('')
    if i == 0:
        ax.set_ylabel('LR pair', fontsize=25)
    else:
        ax.set_ylabel('')
    ax.set_yticklabels(ax.get_yticklabels(), rotation='horizontal', fontsize=20)
    ax.set_xticklabels(ax.get_xticklabels(), rotation='vertical', fontsize=20)

fig.tight_layout()
plt.show()


## Supplementary figure S6 

In [13]:
color = "spatial_domains_sorted"
domains = adata_.obs[color].sort_values().unique()


fig,axes = plt.subplots(ncols=8,nrows=8,figsize=(4*8,5*8))
i=-1
for sample in samples_map.values():
    ncols=len(domains)
    adata = adata_[adata_.obs['sample']==sample]
    for domain in range(8):
        for _ in [0]:
            i += 1
            ax = axes[i//8,i%8]
            if domain == 8:
                ax.set_axis_off()
                continue
            for spine in ax.spines.values():
                spine.set_visible(False)
            mask = space_plot_mask(adata,color,domain,ax=ax,snm=5,sm=50)
            ax.set_title('',fontsize=20)        
            ax.set_xlabel(f"SD{domain}",fontsize=40,color=adata.uns[f'{color}_colors'][domain])
            ax.set_ylabel(sample if i%8 == 0 else '',fontsize=40)
            ax.get_legend().remove()
fig.tight_layout()

i=-1
for sample in samples:
    i += 1
    gap = 0.05  
    for ax in axes[i,:3]:  
        p = ax.get_position()
        ax.set_position([p.x0 - gap/2, p.y0, p.width, p.height])
    for ax in axes[i,6:]:   
        p = ax.get_position()
        ax.set_position([p.x0 + gap/2, p.y0, p.width, p.height])
plt.show()

## Supplementary figure S7 

In [ ]:

col1,col2 = ['spatial_domains_sorted','main_cell_type']


for col2 in ['sample','time']:
    fig,axes = get_figs(1,7,5)
    fs = 15



    ax = axes
    proportions1 = get_proportions(adata_,col1,col2,normalize=False)
    print(proportions1.sum(1))
    print(proportions1.shape)
    display(proportions1)
    proportions1 = get_proportions(adata_,col1,col2,normalize=False)
    plot_region_distribution(proportions1,colors = adata_.uns[f'{col2}_colors'] ,ax=ax,fontsize=12,format_value='.2f',threshold=.1,percentage=True)
    for tl in ax.get_yticklabels():
        c = int(tl._text)
        tl.set_color(adata_.uns[f'{col1}_colors'][c])
    ax.set_yticklabels([f'SD{tick._text}' for tick in ax.get_yticklabels()], rotation='horizontal', fontsize=fs) 
    ax.set_xticklabels([f'{xtick._text}' for xtick in ax.get_xticklabels()], rotation='horizontal', fontsize=15) 
    ax.set_title(f'Domain compositions per {col2}',fontsize=fs)
    ax.set_xlabel('',fontsize=fs)



## Supplementary figure S8

In [ ]:
from matplotlib.patches import Patch
layers = 'XC'


fig,axes = get_figs(2,6,5,vert=False)
for i, layer in enumerate(layers):
    ax = axes[i]
    handles = []
    for domain in range(8):
        key = f'{layer}{domain}'
        if key in purity:
            ks = purity[key]['ks']
            arr = np.array(list(purity[key]['domain_purities'].values())) 
            mean = arr.mean(axis=0)
            color = adata_.uns['spatial_domains_sorted_colors'][domain]
            ax.plot(ks, mean, color=color)
            handles.append(Patch(facecolor=color, label=f'SD{domain}'))
    if i == 1 :
        legend = ax.legend(handles=handles, fontsize=20, bbox_to_anchor=(1.05, 1), loc='upper left')
    ax.set_ylim(0, 1)
    ax.set_title(f'Temporal concentration in {layer}',fontsize=15)
    ax.set_xticklabels(ax.get_xticklabels(), rotation='horizontal', fontsize=15) 
    ax.set_yticklabels(ax.get_yticklabels()if i == 0 else [], rotation='horizontal', fontsize=15) 
    ax.set_xlabel('$n$ neighbors',fontsize=15)
    ax.set_ylabel('Purity score' if i==0 else '',fontsize=15)
    fig.tight_layout()


## Supplementary figure S9

In [ ]:
layer = 'E'
groups = [0,1,2,6],[3,4,5,7]
fs=25

for i,group in enumerate(groups):
    fig,axes = get_figs(4,4,5)
    for ax,domain in zip(axes,group):

        adata = adata_[adata_.obs['spatial_domains_sorted']==domain]
        adata1 = adata1s[f'{domain}']
        umap_plot(adata1,color='time',ax=ax,umap_key=f'{layer}{domain}_umap')
        ax.set_title(f'SD{domain}',fontsize=fs,color=adata_.uns[f'spatial_domains_sorted_colors'][domain])
        ax.get_legend().remove()
        leg = ax.get_legend()                      
        handles = [Patch(facecolor=np.atleast_2d(h.get_facecolor())[0],
                        label=t.get_text())
                for h, t in zip(leg.legend_handles, leg.get_texts())]
        ax.legend(handles=handles, fontsize=fs, bbox_to_anchor=(1.05, 1), loc='upper left')
        ax.set_ylabel('E umap' if domain in [0,3] else '',fontsize=fs)
        ax.set_xlabel('')
        for spine in ax.spines.values():
            spine.set_visible(False)
    fig.tight_layout()
    gap = 0.05  
    for ax in axes[:3]:  
        p = ax.get_position()
        ax.set_position([p.x0 - gap/2, p.y0, p.width, p.height])
    for ax in axes[6:]:   
        p = ax.get_position()
        ax.set_position([p.x0 + gap/2, p.y0, p.width, p.height])

## Supplementary figure S22 

In [ ]:

no_preprocessing = {'preprocessing':None}
preprocessing = {'preprocessing':'image'}
embedding = {"sigma_e":1, "snn":20, 'alpha':0.5}
niche,me,communication=True,True,True

fig,ax0 = get_figs(1)
print('spatial domains')
for n_domains in [2,3,4,5,6,7,8,9,10,11]:
    spatial_domains = {'clustering':'gmm','seed':0,'n_domains':n_domains}
    ko = Kontext(**embedding,verbose=True)
    ko.spatial_domains_of_merged_adata(adata_,**spatial_domains)
    space_plot(adata_,color='spatial_domains',ax=ax0)

    fig,axes = get_figs(n_domains+1,5,5)
    col1,col2 = ['spatial_domains','main_cell_type']
    fs = 25


    ax = axes[0]
    proportions1 = get_proportions(adata_,col1,col2,normalize=False)
    print(proportions1.sum(1))
    print(proportions1.shape)
    display(proportions1)
    proportions1 = get_proportions(adata_,col1,col2,normalize=False)
    plot_region_distribution(proportions1,colors = adata_.uns[f'{col2}_colors'] ,ax=ax,fontsize=12,format_value='.2f',threshold=.1,percentage=True)
    ax.get_legend().remove()
    ax.set_yticklabels([f'SD{tick._text}' for tick in ax.get_yticklabels()], rotation='horizontal', fontsize=fs) 
    ax.set_xticklabels([f'{xtick._text}' for xtick in ax.get_xticklabels()], rotation='horizontal', fontsize=15) 
    ax.set_title('Domain compositions',fontsize=fs)
    ax.set_xlabel('',fontsize=fs)


    domains = range(n_domains)
    sample = '7D1'
    ncols=len(domains)
    for domain,ax in zip(range(n_domains),axes[1:]):
            adata = adata_[adata_.obs['sample']==sample]
            space_plot(adata,color='spatial_domains',ax=ax0)
            print(adata.uns['spatial_domains_colors'])
            for spine in ax.spines.values():
                spine.set_visible(False)
            color = 'spatial_domains'
            mask = space_plot_mask(adata,color,domain,ax=ax,snm=5,sm=50)
            ax.set_title('',fontsize=20)        
            ax.set_xlabel(f"SD{domain}",fontsize=40,color = adata.uns['spatial_domains_colors'][domain])
            ax.set_ylabel("")
            ax.get_legend().remove()
    fig.tight_layout()
    plt.show()



## Pie plot 

In [ ]:

print(ts,tr)
fig, axes = get_figs(2, 5, 5)
for ax, t in zip(axes, [ts, tr]): pie_region(adata_, t, f'sample', 'main_cell_type', ax=ax)
fig, axes = get_figs(2, 5, 5)
for ax, t in zip(axes, [ts, tr]): pie_region(adata_, t, f'time', 'main_cell_type', ax=ax)
fig, axes = get_figs(2, 5, 5)
for ax, t in zip(axes, [ts, tr]): pie_region(adata_, t, f'spatial_domains', 'main_cell_type', ax=ax)

## Umap Plot 

In [ ]:
adata_
pca_of_mat(adata_,adata_.X,n_comps=50,key='X')
adata_.obsm[f'X_umap'] = UMAP(n_components=2).fit_transform(adata_.obsm['X_pca'])
print('C_umap')
adata_.obsm[f'C_umap'] = UMAP(n_components=2).fit_transform(adata_.obsm['C_pca'])
print('E_umap')
adata_.obsm[f'E_umap'] = UMAP(n_components=2).fit_transform(adata_.obsm['E_pca'])


for layer in 'XCE':
    fig,axes = get_figs(4,4,5)
    for i,col in enumerate(['main_cell_type', 'spatial_domains_sorted', 'sample', 'time']):
        ax = axes[i]
        umap_plot(adata_,col,umap_key=f'{layer}_umap',ax=ax,s=.5)
        ax.get_legend().remove()
        ax.set_ylabel('')
        ax.set_xlabel('')